<a id="top"></a>
# Accessing Roman Catalogs: Querying Sky Regions in User-Specified Coordinate Frame
***
## Learning Goals

By the end of this tutorial, you will:

- Understand how to use `astropy regions.SphericalSkyRegion` region instances to define a selection region in an arbitrary coordinate frame. 
- Understand how to use the `FrameTransformerHelper` helper class included with this notebook to conveniently obtain coordinate-transformed query syntax from a `SphericalSkyRegion` instance, and post-process the results table to only include objects within the selected region (in the original frame) and to add columns containing the original frame spatial coordinates.

### Table of Contents
1. [Introduction](#Introduction)
1. [Imports](#Imports)
1. [Performing Spatial Catalogs Selections with Changes of Coordinate Frame](#Performing-Spatial-Catalogs-Selections-wit-Changes-of-Coordinate-Frame)
    1. [Identifying blue stars on the Galactic plane](#Identifying-blue-stars-on-the-Galactic-plane)
    1. [Using astroquery](#Using-astroquery)
        1. [Defining a subset selection region](#Defining-a-subset-selection-region)
        2. [Using the helper class](#Using-the-helper-class)
        3. [Obtaining transformed query spatial constraints](#Obtaining-transformed-query-spatial-constraints)
        4. [Constructing and submitting the query](#Constructing-and-submitting-the-query)
        5. [Using the helper class to transform and trim the results](#Using-the-helper-class-to-transform-and-trim-the-results)
        6. [Visualizing the subset region objects](#Visualizing-the-subset-region-objects)
    1. [Using TAP](#Using-TAP)
        1. [Querying over the full Galactic plane using TAP](#Querying-over-the-full-Galactic-plane-using-TAP)
        2. [Visualizing TAP results across the full Galactic plane](#Visualizing-TAP-results-across-the-full-Galactic-plane)
1. [Additional Resources](#Additional-Resources)
1. [About This Notebook](#About-this-Notebook)

## Introduction

This notebook demonstrates how to translate selection regions of interest defined in a non-ICRS frame, translate that region on the sky to ICRS (the typical coordinate frame used to specify object coordinates in MAST's and other archives' catalogs), perform a spatial query, and then ensure the results catalog is populated with the longitude and latitude information for our non-ICRS frame.

<!-- This notebook demonstrates how to use coordinate transformations to select objects in regions defined in a user-specified coordinate frame that is *not* ICRS (e.g., RA/Dec) --- the typical coordinate frame used to specify object coordinates in MAST's (and other archives') catalogs. -->

As an example, this tutorial demonstrates how to query the MAST PanSTARRS DR2 catalog for objects within regions defined in the Galactic coordinate frame, leveraging the `astropy regions` package to transform the selection region from Galactic to ICRS coordinates, peforming a catalog query, and then post-processing to add Galactic coordinates to the results table (via an included helper class).

While some catalogs already provide Galactic coordinates ($\ell, b$), this generalized approach supports:
* Selection regions defined in arbitrary, user-specified coordinate frames (such as the Heliocentric spherical coordinate system for the orbit of the GD1 stream described in Koposov et al. 2010, [as provided by the gala package](https://gala-astro.readthedocs.io/en/latest/_modules/gala/coordinates/gd1.html)).
* Potential performance improvements by transforming the query to use ICRS coordinates, even if the coordinate frame longitude and latitude are columns (if the database is not spatially indexed on the coordinate in questions, while ICRS position columns are).

## Imports
This tutorial makes use of the following libraries: 
- [*numpy*](https://numpy.org/) for numerical calculations
- [*astropy.coordinates Angle*](https://docs.astropy.org/en/stable/coordinates/index.html) for defining angles
- [*astropy.units*](https://docs.astropy.org/en/stable/units/index.html) to specify angular units
- [*astroquery.mast Catalogs*](https://astroquery.readthedocs.io/en/latest/mast/mast.html) for querying the MAST database
- [*pyvo*](https://pyvo.readthedocs.io) for querying the MAST catalogs via TAP (extended example)
- [*matplotlib.pyplot*](https://matplotlib.org/stable/api/pyplot_summary.html#module-matplotlib.pyplot) for plotting data
- [*regions RangeSphericalSkyRegion*](https://github.com/astropy/regions/tree/main) to define spherical sky regions and handle coordinate frame transformations
- A simple *FrameTransformerHelper* class to prepare catalog queries and post-process the results (adding longitude/latitude columns for the specified coordinate frame).
- [*astropy.table*](https://docs.astropy.org/en/stable/table/index.html) to combine and manipulate result tables
- *time, datetime* to time queries.

In [ ]:
%matplotlib inline

import numpy as np
from astropy.coordinates import Angle
import astropy.units as u
from astroquery.mast import Catalogs
from astropy import table
import pyvo as vo
import matplotlib.pyplot as plt
import time
import datetime

from regions import RangeSphericalSkyRegion

# Helper classes for handling coordinate transformations with queries:
from frame_transformer_helper import FrameTransformerHelper

***

## Performing Spatial Catalogs Selections with Changes of Coordinate Frame

In this notebook, we will construct a sample of bright ($i<18$), blue ($g-y < 0.8$) stars located along the Galactic plane, combining color, magnitude, and spatial constraints, using PanSTARRS DR1 catalogs.

We will demonstrate how to perform this query using both (1) [`astroquery`](https://astroquery.readthedocs.io/en/latest/mast/mast.html) and (2) MAST's [Table Access Protocol (TAP)]("https://mast.stsci.edu/vo-tap/") service using the [Astronomical Data Query Language (ADQL)](http://www.ivoa.net/documents/latest/ADQL.html).


### Identifying blue stars on the Galactic plane

After consulting the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs), we decide to use the 
`MeanObject` table.
This table provides the relevant photometric (columns `gMeanPSFMag`, `iMeanPSFMag`, `yMeanPSFMag`, optimized for point sources; and `iMeanKronMag`, used to distinguish between stars and galaxies) and position (`raMean`, `decMean`) information.

We define our sample selection as:
* Located within $\pm$0.1 degree of the Galactic plane ($|b| \leq 0.1$deg)
* Have $i$ band magnitudes brighter than 18
* Have $g-y$ colors less than 0.8  (the bluest, reddest filters for PanSTARRS)
* Are point sources (not extended objects; point sources typically have `iMeanPSFMAG-iMeanKronMag <= 0.05`, as presented in the [PanSTARRS documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812369/How+to+separate+stars+and+galaxies#Howtoseparatestarsandgalaxies-PSF-Kron)).

We additionally apply quality cuts to ensure flux is detected in the $g$, $i$, $y$ bands (magnitudes > -999, the missing value), and there are at least 3 detection in each of these bands (ng, ni, ny > 3; to avoid data artifacts).

### Using astroquery

To begin, we will implement this catalog search using `astroquery`.

We will begin by defining a small, test region of a limited $(\ell,b)$ range, as we can include only a subset of these constraints in the database query. The remaining constraints --- the color cut $g-y<0.8$, and removing extended sources --- will need to be applied in-memory after obtaining the initial results table.

#### Defining a subset selection region

We begin by defining a small subset, covering a limited range of $\ell=[100,101]$ deg, and $b=[-0.1,0.1]$ deg, using the `RangeSphericalSkyRegion` class. Note that we must specify the coordinate frame, as either an astropy `BaseCoordinateFrame` instance or a string indicating an astropy-registered coordinate frame.

In [ ]:
# Subset selection region:
lon_range = np.array([100, 101])*u.deg
lat_range = np.array([-0.1, 0.1])*u.deg

sel_region = RangeSphericalSkyRegion(
    frame="galactic", 
    longitude_range=lon_range,
    latitude_range=lat_range, 
)
sel_region

#### Using the helper class

We then create an instance of the `FrameTransformerHelper` class, which we will use to 
help prepare our transformed query constraints and ensure the results are translated back to our coordinate frame (for this example, Galactic coordinates).

We specify our selection spherical sky region and our target coordinate frame for our query (ICRS).

In [ ]:
transf_helper = FrameTransformerHelper(
    sel_region=sel_region, 
    frame_target="icrs"
)

This transformation helper contains a property `sel_region_target` that represents the transformation of our region into the ICRS coordinate frame:

In [ ]:
transf_helper.sel_region_target

This is a convenience method for other internal helper processing, that is equivalent to transforming the original region directly:

In [ ]:
sel_region

In [ ]:
sel_region.transform_to("icrs")

#### Obtaining transformed query spatial constraints

We now need to use this transformed region to determing our spatial constraints. 

Currently, the `astroquery.mast Catalogs` module only supports cone searches. Thus, we will need to use the bounding circle of this transformed region.

Our helper class can provide specifically formatted spatial constraints using the `get_bounds_constraints()` method. The arguments specify the search type (bounding circle, 
bounding lon/lat, polygon), the output format (either "ADQL" or "astroquery.mast"), 
and the column names for longitude, latitude in the database table ("keys_pos_target"; for the PanSTARRS mean object table, these are `raMean`, `decMean`

We now use this method to obtain the bounding circle constraints for use with astroquery.

In [ ]:
spatial_constraints = transf_helper.get_bounds_constraints(
    search_type="bound_circle", output_format="astroquery.mast", 
)
spatial_constraints

#### Constructing and submitting the query

We now submit the full query using the `Catalogs.query_criteria()` method, as we will be using a mix of spatial and non-spatial filters. We then apply the post-processing cuts (as noted above) on the preliminary table, to obtain our final sample in this subset region.

In [ ]:
# Submit query:
start = time.time()
results = Catalogs.query_criteria(
    catalog="Panstarrs",
    data_release="dr2",
    table="mean",
    **spatial_constraints, 
    columns=["objID", "raMean", "decMean",
             "gMeanPSFMag", "iMeanPSFMag", "yMeanPSFMag",
             "iMeanKronMag"],    # List of columns to return
    iMeanPSFMag=[("lt", 18)],    # i < 18; i band constraint
    gMeanPSFMag=[("gt", -999)],   # mag > -999; missing magnitudes are -999
    yMeanPSFMag=[("gt", -999)],   # mag > -999; missing magnitudes are -999
    iMeanKronMag=[("gt", -999)],  # mag > -999; missing magnitudes are -999
    ng=[("gt", 3)],   # ng > 3; Number of detections in g
    ni=[("gt", 3)],   # ni > 3; Number of detections in i
    ny=[("gt", 3)],   # ny > 3; Number of detections in y
)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

# Apply post-process cuts:
# 1. Distinguish stars from extended objects using
#    difference of PSF, Kron magnitudes
results = results[
    (results['iMeanPSFMag'] - results['iMeanKronMag']) <= 0.05
]

# 2. Apply color cut
# Copy original catalog for reference:
results = results[
    (results['gMeanPSFMag'] - results['yMeanPSFMag']) < 0.8
]
len(results)

#### Using the helper class to transform and trim the results

Finally, we will use the helper class to post-process the returned table using the helper's `parse_results()` method:
1. As we used a bounding circle for our query, we need to remove entries outside our specified region.
1. To easily work with our results, the helper class will also add derived columns containing the transformed coordinates. **This requires the database position columns to be specified.** Here, we set `keys_pos_target=["raMean", "decMean"]`.


By default, the names for the derived columns will be inferred from the coordinate frame longitude/latitude names. However, custom names can be specified using `keys_pos_orig` (e.g., `keys_pos_orig=["l_derived", "b_derived"]`).  The default units of the output coordinates will be degrees; this can be customized by setting `units_orig` to some other astropy angular unit.

In [ ]:
results = transf_helper.parse_results(
    results,
    keys_pos_target=["raMean", "decMean"]
)
results

Our final sample in this subset region is small (41 objects).

#### Visualizing the subset region objects

The targets selected in this subset Galactic longitude/latitute range are shown below in the ICRS (database) and Galactic (selection) frames:

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(9, 3))

axes[0].set_title("ICRS")
axes[0].set_xlabel("RA")
axes[0].set_ylabel("Dec")
axes[0].grid(True)
axes[0].scatter(
    results["raMean"], results["decMean"],
    marker=".", s=50, lw=0
)

axes[1].set_title("Galactic")
axes[1].set_xlabel(r"$\ell$")
axes[1].set_ylabel(r"$b$")
axes[1].grid(True)
axes[1].scatter(
    results["l"], results["b"],
    marker=".", s=50, lw=0
)

fig.subplots_adjust(top=0.95, bottom=0.0, wspace=0.3)

plt.show()

As this individual chunk query took about 30 seconds, it would take about 2.5 hours query over all chunks; we leave iterating over all chunks as an exercise to interested readers.

However, MAST's PanSTARRS TAP service runs on a much more powerful database, making it possible to much more quickly iterate over all subsets to obtain the full sample across the whole Galactic plane, as shown below.

### Using TAP

We now demonstrate how to implement this search using [ADQL](http://www.ivoa.net/documents/latest/ADQL.html) to query MAST's PanSTARRS [TAP](https://mast.stsci.edu/vo-tap/) service.

This advanced query patterns possible with ADQL streamline this search, enabling us to avoid in-memory post-filtering (for the color and star/extended object cuts).

However, as currently this TAP service only supports cone searches, we will still need to chunk over multiple longitude ranges.

#### Querying over the full Galactic plane using TAP

In this case, we'll proceed directly iterating over all longitude ranges.

First, we connect to the MAST PanSTARRS DR2 TAP service.

In [ ]:
# Use `pyvo` to connect to the MAST PanSTARRS TAP service:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs/"
)

Again, we will access the mean object catalog (`mean_object` here; see the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs)), which contains all the columns we will need.

(*Note:* this TAP service is backed by a new, more powerful database. The column names in this database are entirely lowercase, but are otherwise equivalent to the columns of the legacy database as documented above.)

We now construct our query, including the color/magnitude constraints and 
star/extended object separation. As an overview, our query is constructed as follows.

The table, `ps1_dr2.mean_object`, is specified using the "FROM" statement.
We specify the coluns to be returned (`objid`, `ramean`, `decmean`, `gmeanpsfmag`, `imeanpsfmag`, `ymeanpsfmag`, `imeankronmag`) using the "SELECT" statement.
The query constraints are specified in the "WHERE" statement, with multiple constraints linked using "AND". 
This includes:
* the cone search (as in the "CONTAINS(...)" clause; note that for ADQL, the radius must be expressed in degrees), 
* requiring the $g$, $i$, $y$ mean PSF magnitudes and $i$ mean Kron magnitude to be present (cutting objects that have any of these values missing),
* the PanSTARRS-specific cut to select stars (`imeanpsfmag - imeankronmag <= 0.05`), and 
* the $g-y$ color and $i$ magnitude cuts.

As before, for each longitude chunk, we (1) define the subset region, (2) leverage the helper to define the ADQL-specific spatial constraint syntax, (3) integrate this into the full query, and (4) query the database and obtain the result, and (5) cut back to our selection region using the helper.

Because we can perform all non-spatial filtering server-side, we use wider longitude ranges (5 degrees) in this case (though still relatively small to ensure no truncation of returned rows).

In [ ]:
# Selection regions:
# Same latitude range for all chunks
lat_range = np.array([-0.1, 0.1])*u.deg

# Prepare aggregate table:
results_tap = None

start = time.time()
for lon in range(72):
    # Define subset longitude range:
    lon_range = np.array([lon, lon+1])*5*u.deg

    # Define subset spherical region:
    sel_region = RangeSphericalSkyRegion(
        frame="galactic", 
        longitude_range=lon_range,
        latitude_range=lat_range, 
    )

    # Define transform helper:
    transf_helper = FrameTransformerHelper(
        sel_region=sel_region, 
        frame_target="icrs"
    )

    # Obtain query spatial constraints:
    spatial_constraints = transf_helper.get_bounds_constraints(
        search_type="bound_circle", output_format="ADQL", 
        keys_pos_target=["ramean", "decmean"],
    )

    # Specify query
    # Note: ADQL comments are preceded by "--"
    adql_query = f"""
    SELECT objid, ramean, decmean, 
        gmeanpsfmag, imeanpsfmag, ymeanpsfmag, 
        imeankronmag
    FROM ps1_dr2.mean_object
    WHERE {spatial_constraints}                -- search constraints from the helper
    AND imeanpsfmag < 18                      -- magnitude cut
    AND (gmeanpsfmag-ymeanpsfmag) < 0.8       -- color cut
    AND (imeanpsfmag - imeankronmag) <= 0.05  -- distinguish stars from extended objects
    AND gmeanpsfmag > -999                    -- mag > -999; missing magnitudes are -999
    AND ymeanpsfmag > -999                    -- mag > -999; missing magnitudes are -999
    AND imeankronmag > -999                   -- mag > -999; missing magnitudes are -999
    """
    
    # Submit query to the MAST TAP service:
    job = TAP_service.run_async(adql_query)
    
    # Retrieve results:
    results_tap_chunk = job.to_table()

    # Parse results & restrict to original region:
    results_tap_chunk = transf_helper.parse_results(results_tap_chunk)

    # Combine with the full results set:
    if results_tap is None:
        results_tap = results_tap_chunk
    else:
        results_tap = table.vstack([results_tap, results_tap_chunk])

end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

# Ensure all entries are unique:
results_tap = table.unique(results_tap, keys="objid")
results_tap

#### Visualizing TAP results across the full Galactic plane

We now display our final sample using a full-sky Aitoff projection, first in ICRS (left) and then Galactic (right) coordinates.

(Note that the columns from the TAP results table have units.)

In [ ]:
fig, axes = plt.subplots(
    ncols=2,
    figsize=(15, 5),
    subplot_kw=dict(projection="aitoff")
)

axes[0].set_title("ICRS, Aitoff projection", pad=15)
axes[0].set_xlabel("RA")
axes[0].set_ylabel("Dec")
axes[0].grid(True)
axes[0].scatter(
    Angle(results_tap["ramean"]).wrap_at(180 * u.deg).radian, 
    Angle(results_tap["decmean"]).radian, 
    marker=".", s=5, lw=0
)

axes[1].set_title("Galactic, Aitoff projection", pad=15)
axes[1].set_xlabel(r"$\ell$")
axes[1].set_ylabel(r"$b$")
axes[1].grid(True)
axes[1].scatter(
    Angle(results_tap["l"]).wrap_at(180 * u.deg).radian, 
    Angle(results_tap["b"]).radian, 
    marker=".", s=5, lw=0
)

fig.subplots_adjust(top=0.95, bottom=0.0)

We can see the observational limitation of PanSTARRS (observing only Decl > -30 degrees) clearly in the left panel (showing ICRS), while the Galactic coordinates of the right panel very closely follow the Galactic midplane (as expected given our spatial constraint).

--------

## Additional Resources

### Astropy Coordinate Transformations
- The `astropy.coordinates` documentation provides full details about [transforming between coordinate frames](https://docs.astropy.org/en/stable/coordinates/transforming.html) using astropy classes.
- Custom coordinate frames can be defined for use with the `astropy.coordinates` functionality. To support transformations between custom frames and other frames such as ICRS, it is also necessary to define a transformation to transform between the custom frame and (at least) one of frames included in `astropy.coordinates`. See [the `astropy.coordinates` documentation](https://docs.astropy.org/en/stable/coordinates/frames.html#defining-transformations-between-frames) for full details.

### Astroquery.mast

- Access to MAST holdings, including catalogs, through the [Astropy-affiliated astroquery package](https://astroquery.readthedocs.io/en/latest/mast/mast.html).
- A full list of available MAST Astroquery catalog services is available through the [astroquery documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_catalog.html).

### Table Access Protocol (TAP) & the Astronomy Query Data Language (ADQL)
- [Documentation for `pyvo`](https://pyvo.readthedocs.io/en/latest/index.html), a package to retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- [Documentation on ADQL](http://www.ivoa.net/documents/latest/ADQL.html), covering the specifications of this IVOA standard for querying astronomical data in tabular format, with geometric search support.
- [Documentation on TAP](http://www.ivoa.net/documents/TAP/), the IVOA standard for RESTful web service access to tabular data.
- [Full list of TAP services at MAST](https://mast.stsci.edu/vo-tap).


### PanSTARRS 1 DR 2

- See the [PanSTARRS documentation](https://outerspace.stsci.edu/display/PANSTARRS/) for full details regarding PanSTARRS DR2 catalogs.


## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)


## About This Notebook

**Authors:** Sedona Price<br>
**Keywords:** Tutorial, coordinate frames, astroquery, TAP<br>
**Last updated:** June 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 